## Word-2-vector

In [3]:
## 1.
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
# Compare query with the document

v1.dot(dv)

np.float32(0.32332397)

In [7]:
vc = v1.copy()
v1.dot(vc) # Returns 1.0 because they are the same vector

np.float32(0.9999999)

In [ ]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv) # This has a low similarity

np.float32(0.019730574)

## Dataset embeddings

In [7]:
## 2.
from ingest import load_faq_data

documents = load_faq_data()

In [8]:
## 3.
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [9]:
## 4.
from tqdm.auto import tqdm

# We encode the documents in batches to avoid memory issues with large datasets
batch_size = 50
vectors = []

# Create embeddings for all the documents
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [6]:
## 5.
import numpy as np

# Turn vectors into a matrix
X = np.array(vectors)

In [9]:
X.shape

(1350, 384)

## Manual Vector Search

In [10]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [ ]:
# Compare query with all documents matrix
# Returns cosine similarity scores for each document
scores = X.dot(v_query)

# Alternatively - calculate it in a loop (not recommended)
# scores = [v_query.dot(X[i]) for i in range(len(X))]

In [13]:
# Best match
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [14]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [ ]:
# Top 5 results - argsort sorts from lowest to highest (and returns the indices)
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1] # Reverse to get 5 highest to lowest
top5

array([  2, 625, 907, 538,   7])

In [ ]:
# Top 5 scores
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [18]:
# Simplified way of getting the top 5 - reverse the scores first
top5 = np.argsort(-scores)[:5]
top5

array([  2, 625, 907, 538,   7])

In [ ]:
# Top 5 documents from a vector search
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

## Minsearch Vector Search

In [ ]:
from minsearch import VectorSearch

# Initialize and fit vector index
# Minsearch allows for keyword filtering - filter out documents before vector search
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [14]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [ ]:
# Top 5 scores without filtering
results[:5]

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

In [17]:
# Include keyword filtering
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [18]:
# Top 5 scores with filtering
results[:5]

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

## RAG with Vector Search

In [20]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
# wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-26 10:12:41--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-26 10:12:42 (37.4 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [21]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents) # Text search index

In [23]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [32]:
# First we try with text search for comparison
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [35]:
assistant.rag("How to run Olama locally?")

'To run Ollama locally:\n\n1. Install Ollama from: https://ollama.com/download  \n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Start a model locally:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a chat-like interface.\n\n3. Test that the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a JSON response.\n\n4. If you want to use it from Python:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": your_prompt}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\n'

In [27]:
# Replace text search with vector search
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder # Model used to create embeddings

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex, # This time we use vector index
    llm_client=openai_client,
)

In [29]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want a certificate, make sure you submit your project while submissions are still open.'

In [30]:
vector_assistant.rag("How to run Olama locally?")

'To run Ollama locally:\n\n1. Install Ollama from **https://ollama.com/download**\n   - macOS: install the `.pkg`\n   - Windows: install the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Start a model locally:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a chat-like local interface.\n\n3. Test that the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. If you want to use it from Python:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": your_prompt}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you meant something else by “Olama,” I don’t know.'

## Sqlitesearch Vector Search - persistent index

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [10]:
vs_index.fit(vectors, documents) # Save index to db

In [11]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [12]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

In [13]:
# Filter by course

results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [14]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [15]:
vs_index.close()